# Setup: turn a stock image into the repo-intel GPU image

Run this **once**, in a Jupyter job on the SVKM cluster. At the end you save the
container as your own image and never run this again — later jobs start from the
saved image and go straight to work.

**Start the job like this** in Altair Access → Applications → Jupyter:

| Field | Value |
|---|---|
| Container Image | `pytorch_pbs:25.12-py3` |
| Number of Nodes | 1 |
| Number of Processors per Node | **8** (not 1) |
| Number of GPUs per Node | 1 |
| Amount of Memory (MB) | **32000** (not 10) |
| Queue | `workq` |

The two bolded fields default to values that will kill the job: 1 CPU and **10 MB**
of RAM. Set them before submitting.

### Which base image

`pytorch_pbs:25.12-py3` is the newest on the cluster and is what to try first.
We use it for Python and Jupyter, **not** for PyTorch — inference runs through
Ollama, which ships its own CUDA runtime and talks to the host driver directly.
Nothing in this project imports torch at run time (BERTScore is the only thing that
would, it is lazily imported, and it is switched off).

That also means the container's CUDA version barely matters to us. If Ollama turns
out not to see the GPU on 25.12, fall back to **`pytorch_pbs:23.06-py3`**, which is
built against an older CUDA that pairs with the driver shown in the manual
(535.86.10 / CUDA 12.2). Do not read anything into `torch.cuda.is_available()`
being False — we never use torch. Section 5's Ollama check is the one that counts.


## 1. What did we actually get?

Run this first. It answers the three things the manual does not: how much GPU we
were given, whether this node can reach the internet, and where to put files.


In [ ]:
import os, shutil, socket, subprocess, sys

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=60).stdout.strip()
    except Exception as e:
        return f'(failed: {e})'

print('=== GPU ===')
print(sh('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader') or '(no nvidia-smi)')
mig = sh("nvidia-smi --query-gpu=mig.mode.current --format=csv,noheader")
print('MIG mode:', mig or '(unknown)')
print(sh('nvidia-smi -L'))

print('\n=== CPU / RAM / disk ===')
print('cores visible :', os.cpu_count())
mem = sh("grep MemTotal /proc/meminfo")
print('host memory   :', mem)
for p in ('/data', '/tmp', os.path.expanduser('~')):
    if os.path.isdir(p):
        t, u, f = shutil.disk_usage(p)
        print(f'{p:<8} free {f/2**30:6.1f} GB of {t/2**30:6.1f} GB')

print('\n=== network (decides whether we can pull models here) ===')
for host, port, label in [('pypi.org',443,'PyPI (pip)'),
                          ('github.com',443,'GitHub (git clone)'),
                          ('registry.ollama.ai',443,'Ollama model registry')]:
    try:
        socket.create_connection((host, port), timeout=6).close()
        print(f'  REACHABLE  {label}')
    except Exception as e:
        print(f'  BLOCKED    {label}  ({type(e).__name__})')


### How to read that

- **GPU** should show an H100 and, under MIG, a ~40 GB slice.
  - Battery jobs: worst case is `qwen2.5-coder:32b` (~20 GB) + `gemma2:9b` (~6 GB) = ~26 GB — comfortably under the 40 GB limit.
  - Rejudge jobs: `gemma2:27b` alone is ~17 GB (42% of slice). Both fit with headroom.
- **Ollama model registry REACHABLE** → you can pull models here (§4).
  **BLOCKED** → you must upload the weights instead; §4 explains that path.
- **`/data`** is your persistent storage and survives between jobs. `/tmp` usually
  does not.


## 2. Install Ollama

We keep Ollama rather than running the models through PyTorch directly. It serves
**4-bit quantized** weights, which is what every result so far was produced with.
Switching to fp16 would use the H100 better and produce *different text from the
same model* — a different experiment, under which none of the existing 157 summaries
stay comparable. See `docs/GPU_BATCH.md`.


In [ ]:
# The PBS job scheduler sets $TMPDIR to a job-specific path that does not exist
# inside the container. The Ollama install script uses mktemp internally, which
# reads $TMPDIR and fails. Override it to /tmp before running the installer.
import os
os.environ['TMPDIR'] = '/tmp'

# The installer needs GitHub. If §1 showed GitHub BLOCKED, skip to the manual
# fallback in the next cell instead.
!TMPDIR=/tmp curl -fsSL https://ollama.com/install.sh | sh

# Confirm the binary landed in PATH
!which ollama && ollama --version


In [ ]:
# If 'ollama --version' above printed 'command not found', the binary is installed
# but not in PATH for this shell. Force-add the standard install location.
import os
os.environ['PATH'] = '/usr/local/bin:' + os.environ.get('PATH', '')
os.environ['TMPDIR'] = '/tmp'  # keep set for any subsequent shell calls
!which ollama && ollama --version


In [ ]:
# FALLBACK, only if the installer could not reach GitHub.
# Download the tarball somewhere with internet, upload it via Altair's +Upload into
# /data/mpstme-dishank/, then run this.
#
# import os; os.environ['TMPDIR'] = '/tmp'
# !tar -C /usr -xzf /data/mpstme-dishank/ollama-linux-amd64.tgz
# !which ollama && ollama --version


## 3. Python dependencies and the project code

Upload the repository as a zip through Altair (**+Upload**) into `/data/<you>/`, or
clone it if §1 showed GitHub is reachable.


In [ ]:
import os, subprocess
USER = os.environ.get('USER', 'mpstme-dishank')
DATA = f'/data/{USER}'
PROJ = f'{DATA}/repo-intel-platform'
print('project dir ->', PROJ)

if not os.path.isdir(PROJ):
    print('Not found. Either:')
    print(f'  git clone https://github.com/palsoniii/repo-intel-platform {PROJ}')
    print(f'  ...or upload a zip to {DATA} and: unzip repo-intel-platform.zip -d {DATA}')
else:
    print('found')


In [ ]:
# Install the project's Python dependencies.
# NOTE: we deliberately do NOT install or verify torch. bert-score pulls it in, but
# it is lazily imported and BERTScore is disabled for the battery, so it is never
# touched. If pip tries to fetch a large CUDA torch wheel here, that is wasted time
# and disk -- the image already has one, and we do not use it either way.
!pip install --no-cache-dir -r {PROJ}/backend/requirements.txt

# What actually has to work: the harness's own imports, with no GPU involved.
import subprocess
chk = subprocess.run(
    ['python', '-c', 'import app.evaluation.harness; print("harness imports OK")'],
    cwd=f'{PROJ}/backend', capture_output=True, text=True)
print(chk.stdout or chk.stderr[-2000:])


## 4. Stage the models

Roughly **55 GB** total for all models. Put them on `/data` so they persist
between jobs and are **not** baked into the saved image.

VRAM budget:
- Battery job (worst case): `qwen2.5-coder:32b` (~20 GB) + `gemma2:9b` (~6 GB) = ~26 GB — fits in 40 GB slice.
- Rejudge job: `gemma2:27b` (~17 GB) alone — no generator co-resident.


In [ ]:
import os, time
os.environ['OLLAMA_MODELS'] = f'{DATA}/ollama'
os.makedirs(os.environ['OLLAMA_MODELS'], exist_ok=True)

# Hold the generator and the judge in VRAM together. The battery makes two judge
# calls per generation, so with only one model resident every single row pays for
# an evict-and-reload. That thrash dominated the laptop runs; a 40GB slice makes it
# unnecessary. Purely a scheduling change -- weights, context size and decoding are
# identical, so outputs do not change.
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '2'
os.environ['OLLAMA_NUM_PARALLEL'] = '1'   # batching can alter generated text; keep off
os.environ['OLLAMA_KEEP_ALIVE'] = '30m'   # do not evict between rows
os.environ['OLLAMA_NUM_CTX'] = '8192'     # must match the study

!pkill -f 'ollama serve' 2>/dev/null; sleep 1
env = ' '.join(f'{k}={os.environ[k]}' for k in
               ['OLLAMA_MODELS','OLLAMA_MAX_LOADED_MODELS','OLLAMA_NUM_PARALLEL','OLLAMA_KEEP_ALIVE'])
get_ipython().system_raw(f'{env} nohup ollama serve > /tmp/ollama.log 2>&1 &')
time.sleep(5)
!ollama list || tail -20 /tmp/ollama.log


In [ ]:
# Final generator roster. gemma2:9b is the SOLE PRIMARY JUDGE -- never a generator.
#
# codellama:13b-instruct  -- Meta, cross-org comparison anchor (~8 GB Q4_K_M)
# qwen2.5-coder:14b       -- Alibaba, scale-matched vs codellama:13b (~9 GB Q4_K_M)
# qwen2.5-coder:32b       -- Alibaba, two-point within-family check (~20 GB Q4_K_M)
#
# Do NOT substitute qwen3-coder:30b -- different architecture, different training,
# results would not be comparable.
MODELS = ['codellama:13b-instruct', 'qwen2.5-coder:14b', 'qwen2.5-coder:32b']
JUDGE  = 'gemma2:9b'

# Secondary judges -- staged here so rejudge jobs (run_rejudge.sh) can start
# immediately after battery jobs finish. NEVER co-resident with a generator;
# each rejudge job loads only one secondary judge at a time.
#   mistral:7b-instruct: ~5 GB Q4_K_M
#   gemma2:27b:          ~17 GB Q4_K_M  (fits alone in 40 GB slice)
SECONDARY_JUDGES = ['mistral:7b-instruct', 'gemma2:27b']

# Skip this cell if the Ollama registry was BLOCKED in section 1; pull these on a
# machine with internet and upload the whole $DATA/ollama directory instead.
for m in MODELS + [JUDGE] + SECONDARY_JUDGES:
    print(f'--- pulling {m} ---')
    !ollama pull {m}
!ollama list

# qwen2.5-coder:32b is the one to double-check: if the tag does not resolve on
# ollama.com, do NOT substitute another model -- that would break cross-run
# comparability. Report the issue instead.


## 5. Smoke test

One generation, end to end. If this works, the image is good.


In [ ]:
import time, os
os.environ['OLLAMA_HOST'] = 'http://127.0.0.1:11434'
os.environ['OLLAMA_NUM_CTX'] = '8192'   # must match the study, or nothing is comparable

t0 = time.time()
!ollama run qwen2.5-coder:14b "Reply with exactly: OK" --verbose
print(f'\nwall clock: {time.time()-t0:.1f}s')
print('\nGPU during/after the call — memory should be in use, not 0:')
!nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader


### The number that matters

On the RTX 3050 with CPU offload, a real summary averaged **88 seconds**. Fully
resident on a 40 GB slice, expect roughly **5–15× faster**. If your smoke test is
slow and `nvidia-smi` shows little memory in use, the model is running on CPU —
stop and fix that before running a battery, or you will burn hours for nothing.


## 6. Save the container

Back in **Altair Access**:

1. Note this job's ID (e.g. `pbs.1325.srv-svkmmastermum.x8z`)
2. **Custom Actions → Save Docker Container**
3. **Job ID**: this job · **New Image Name**: `repo-intel-gpu` · **Current Working Directory**: `/data`
4. **Run**

Future jobs set *Container Image* to `repo-intel-gpu` and skip this whole notebook.

**Do not put the models or the repo inside the image.** They live on `/data`, which
every job can see. Baking them in makes the image tens of GB and stale the moment
either changes.
